## 🧱 1. Armado del Dataset

Se parte del dataset original de ventas sell-in.

Granularidad: `<product_id, periodo>`

Transformación:
- Agrupar por `product_id`, `periodo`
- Sumarizar `tn` (toneladas)


In [1]:
import pandas as pd

# Cargar el archivo de sell-in (ajustar el path si es necesario)
df_raw = pd.read_csv("sell-in.txt", sep="\t")

# Agrupar por product_id y periodo
df_agg = (
    df_raw.groupby(['product_id', 'periodo'], as_index=False)
           .agg({'tn': 'sum'})
           .sort_values(['product_id', 'periodo'])
)

df_agg.head()


,product_id,periodo,tn
0,20001,201701,934.77222
1,20001,201702,798.01620
2,20001,201703,1303.35771
3,20001,201704,1069.96130
4,20001,201705,1502.20132


In [2]:
df_agg.shape

(31243, 3)

## 🧮 2. Cálculo de la Clase (target)

Se crea un nuevo campo `clase` que representa `tn` en `periodo + 2`.

Notas:
- Para calcularlo se hace un merge desplazando dos períodos hacia atrás por `product_id`.
- Los períodos `201911` y `201912` quedan sin target (NaN).


In [3]:
# Asegurarse de que periodo sea tipo entero
df_agg['periodo'] = df_agg['periodo'].astype(int)

# Crear índice temporal: mes_abs (mes absoluto)
periodos_ordenados = sorted(df_agg['periodo'].unique())
map_periodo_to_mesabs = {p: i + 1 for i, p in enumerate(periodos_ordenados)}

# Agregar columna mes_abs
df_agg['mes_abs'] = df_agg['periodo'].map(map_periodo_to_mesabs)

# Ordenar correctamente por producto y tiempo
df_agg = df_agg.sort_values(['product_id', 'mes_abs'])

# Crear campo tn+2 (la clase) como tn desplazado -2 hacia adelante
df_agg['tn+2'] = df_agg.groupby('product_id')['tn'].shift(-2)

# Revisar el resultado
df_agg.head(40)


,product_id,periodo,tn,mes_abs,tn+2
0,20001,201701,934.77222,1,1303.35771
1,20001,201702,798.01620,2,1069.96130
2,20001,201703,1303.35771,3,1502.20132
3,20001,201704,1069.96130,4,1520.06539
4,20001,201705,1502.20132,5,1030.67391
5,20001,201706,1520.06539,6,1267.39462
6,20001,201707,1030.67391,7,1316.94604
7,20001,201708,1267.39462,8,1439.75563
8,20001,201709,1316.94604,9,1580.47401
9,20001,201710,1439.75563,10,1049.38860


## 🛠️ 3. Feature Engineering: Lags

Se generan 11 columnas de `tn` anteriores (tn_1 a tn_11), por `product_id`.

En cada fila, se tiene el historial de 12 meses completos si está disponible:
- `tn`, `tn_1`, ..., `tn_11`

No se generan features adicionales.


In [4]:
# Crear los lags tn_1 a tn_11
for lag in range(1, 12):
    df_agg[f'tn_{lag}'] = df_agg.groupby('product_id')['tn'].shift(lag)

df_agg.head()


,product_id,periodo,tn,mes_abs,tn+2,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
0,20001,201701,934.77222,1,1303.35771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,201702,798.01620,2,1069.96130,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20001,201703,1303.35771,3,1502.20132,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20001,201704,1069.96130,4,1520.06539,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20001,201705,1502.20132,5,1030.67391,1069.96130,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 🎯 4. Dataset de Entrenamiento

Se selecciona únicamente el período `201812`.

Subset estratégico: solo los 33 `product_id` mágicos con datos completos.

Campos usados:
- Input: `tn`, `tn_1`, ..., `tn_11`
- Target: `clase` (mes `201902`)


In [5]:
# Lista de product_id mágicos
magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
   20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046, 20049,
   20051, 20052, 20053, 20055, 20008, 20001, 20017, 20086, 20180,
   20193, 20320, 20532, 20612, 20637, 20807, 20838]

# Selección de features y target
features = ['tn'] + [f'tn_{i}' for i in range(1, 12)]
target = 'tn+2'

# Filtrar el dataset para entrenamiento: mes_abs == 24 (equivale a periodo 201812)
df_train = df_agg[df_agg['mes_abs'] == 24].copy()

# Quedarse solo con los mágicos
df_train = df_train[df_train['product_id'].isin(magicos)]

# Eliminar registros incompletos
df_train = df_train.dropna(subset=features + [target])

# Mostrar resultados
print(f"Registros para entrenamiento: {df_train.shape[0]}")
display(df_train[features + [target]].head(10))


Registros para entrenamiento: 33


,tn,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11,tn+2
23,1486.68669,1813.01511,2295.19832,1438.67455,1800.96168,1470.41009,1150.79169,1293.89788,1251.28462,1856.83534,1043.76470,1169.07532,1259.09363
59,1009.45458,1766.81068,1378.49032,954.23575,1161.88430,977.40239,1033.82845,1103.39191,999.20934,966.86044,712.00087,984.80167,1043.01349
95,769.82869,1206.91773,1313.34211,912.34156,955.97079,656.22700,660.73323,784.35885,765.47838,778.55594,788.30749,907.56304,758.32657
203,407.75925,566.66809,513.15472,478.04388,615.70617,515.20419,468.15260,865.28861,748.44391,862.19361,588.56272,470.33785,479.99914
275,426.32899,433.50170,532.45644,436.96269,554.82147,526.38149,554.57063,707.59267,691.53246,765.98901,506.25385,469.29224,476.98787
347,285.02947,414.97753,612.50721,480.60235,582.83104,331.96807,223.87746,227.24082,171.74107,653.77607,477.48363,298.25586,337.76009
383,321.09714,289.13976,177.75576,189.59850,191.07270,300.26178,437.75550,484.04538,562.70214,526.99374,601.26066,340.75314,431.62938
599,259.32724,286.83676,331.23254,288.35292,374.95908,351.60065,316.45841,533.53335,550.29417,488.79258,377.84497,291.70926,308.71060
635,326.01506,371.52958,161.58557,282.43485,375.61778,325.03223,420.33781,388.43687,543.06908,510.33171,337.54792,342.16945,265.84135
671,446.69747,532.98143,552.71975,417.95455,387.73155,351.05610,262.33076,356.42982,290.39581,321.26878,629.89543,243.71984,323.66178


## 📈 5. Entrenamiento del Modelo (Regresión Lineal)

Modelo: Regresión Lineal sin hiperparámetros

- X: tn, tn_1, ..., tn_11
- y: clase


In [10]:
from sklearn.linear_model import LinearRegression
import pandas as pd

# entreno 12 modelos, uno usando tn, otro usando tn y tn_1, otro usando tn, tn_1 y tn_2,hasta llegar a tn, tn_1, ..., tn_11
features = ['tn'] + [f'tn_{i}' for i in range(1, 12)]
print(f"Entrenando modelo con features: {features}")
X = df_train[features]
y = df_train['tn+2']
model = LinearRegression()
model.fit(X, y)
coef = pd.DataFrame({
    'feature': ['intercept'] + features,
    'coeficiente': [model.intercept_] + list(model.coef_)
})
coef['abs'] = coef['coeficiente'].abs()
display(coef)


Entrenando modelo con features: ['tn', 'tn_1', 'tn_2', 'tn_3', 'tn_4', 'tn_5', 'tn_6', 'tn_7', 'tn_8', 'tn_9', 'tn_10', 'tn_11']


,feature,coeficiente,abs
0,intercept,0.441467,0.441467
1,tn,-0.001339,0.001339
2,tn_1,0.236558,0.236558
3,tn_2,0.178208,0.178208
4,tn_3,-0.060031,0.060031
5,tn_4,-0.161875,0.161875
6,tn_5,-0.007775,0.007775
7,tn_6,0.151936,0.151936
8,tn_7,0.043933,0.043933
9,tn_8,0.142839,0.142839


## 📊 6. Aplicación del Modelo a los 780 registros finales

- Se aplicará solo a los 656 con datos completos.
- Los 124 restantes se imputan con el promedio.


In [11]:
# Agarramos los 780 productos correspondientes a predecir
df_780 = pd.read_csv("product_id_apredecir201912.txt", sep=';')
df_780.columns = ['product_id']
df_780.shape

(780, 1)

In [12]:
df_agg

,product_id,periodo,tn,mes_abs,tn+2,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
0,20001,201701,934.77222,1,1303.35771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,201702,798.01620,2,1069.96130,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20001,201703,1303.35771,3,1502.20132,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20001,201704,1069.96130,4,1520.06539,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20001,201705,1502.20132,5,1030.67391,1069.96130,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31238,21295,201701,0.00699,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31239,21296,201708,0.00651,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31240,21297,201701,0.00579,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31241,21298,201708,0.00573,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_pred = df_agg[
    (df_agg['mes_abs'] == 36) & 
    (df_agg['product_id'].isin(df_780['product_id']))
].copy()
df_pred

,product_id,periodo,tn,mes_abs,tn+2,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
35,20001,201912,1504.68856,36,NaN,1397.37231,1561.50552,1660.00561,1261.34529,1678.99318,1109.93769,1629.78233,1647.63848,1470.65653,1259.09363,1275.77351
71,20002,201912,1087.30855,36,NaN,1423.57739,1979.53635,1090.18771,813.78215,1066.44999,928.36431,1034.98927,1287.62346,1083.62552,1043.01349,1266.78751
107,20003,201912,892.50129,36,NaN,948.29393,1081.36645,967.77116,635.59563,715.20314,662.38654,590.12515,565.33774,638.04010,758.32657,964.76919
143,20004,201912,637.90002,36,NaN,723.94206,1064.69633,786.17140,482.13372,521.71519,667.19411,603.31081,466.70901,619.77084,441.70332,511.33713
179,20005,201912,593.24443,36,NaN,606.91173,996.78275,879.52808,536.66800,745.74978,876.39696,897.26297,624.99880,488.21387,409.89950,363.58438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31090,21263,201912,0.01270,36,NaN,0.03247,0.01552,0.01128,0.03388,0.03387,0.00988,0.02258,0.01835,0.06636,0.05927,0.04376
31129,21265,201912,0.05007,36,NaN,0.06600,0.10921,0.01707,0.01593,0.02959,0.05121,0.17635,0.36405,0.01593,NaN,NaN
31139,21266,201912,0.05121,36,NaN,0.06713,0.11831,0.02844,0.01480,0.05916,0.05235,0.17634,0.36178,0.01707,NaN,NaN
31149,21267,201912,0.01569,36,NaN,0.04052,0.09676,0.01830,0.04054,0.07452,0.05882,0.24451,0.12291,0.21578,NaN,NaN


In [15]:
predictions = []
# itero sobre los modelos (desde el mas grande al mas chico) y me fijo que datasets son aptos para predecir
# es decir, los que no tienen ningun NaN para esas features, los que ya tienen un modelo, no los vuelvo a predecir
# ordeno modelos de mayor a menor cantidad de lags

# Filtrar df_pred para obtener solo los productos que no han sido predichos aún
df_to_predict = df_pred[df_pred['product_id'].isin(df_780['product_id']) & 
                        df_pred[features].notna().all(axis=1)]



# Predecir
X_pred = df_to_predict[features]
y_pred = model.predict(X_pred)

predictions = pd.DataFrame({
    'product_id': df_to_predict['product_id'],
    'tn': y_pred
})
# Agregar resultados a la lista de predicciones
predictions

,product_id,tn
35,20001,1162.707525
71,20002,1183.640604
107,20003,684.763931
143,20004,580.484961
179,20005,563.560780
...,...,...
30913,21248,0.468061
30985,21256,0.463856
31036,21259,0.467856
31075,21262,0.465820


In [17]:
df_autogluon = df_agg[df_agg["product_id"].isin(df_780["product_id"])]
df_autogluon

,product_id,periodo,tn,mes_abs,tn+2,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
0,20001,201701,934.77222,1,1303.35771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,201702,798.01620,2,1069.96130,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20001,201703,1303.35771,3,1502.20132,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20001,201704,1069.96130,4,1520.06539,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20001,201705,1502.20132,5,1030.67391,1069.96130,1303.35771,798.01620,934.77222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31206,21276,201908,0.01265,32,0.02079,0.00223,0.04086,0.09283,0.10173,0.12249,NaN,NaN,NaN,NaN,NaN,NaN
31207,21276,201909,0.01856,33,0.03341,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,NaN,NaN,NaN,NaN,NaN
31208,21276,201910,0.02079,34,0.00892,0.01856,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,NaN,NaN,NaN,NaN
31209,21276,201911,0.03341,35,NaN,0.02079,0.01856,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,NaN,NaN,NaN


In [18]:
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

df_autogluon["timestamp"] = pd.to_datetime(df_autogluon["periodo"].astype(str), format='%Y%m')
df_autogluon["target"] = df_autogluon["tn+2"]
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_autogluon,
    id_column="product_id",
    timestamp_column="timestamp",
)
ts_data = ts_data.fill_missing_values()
predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="target",
    freq="MS"
)
predictor.fit(ts_data, num_val_windows=2)

/tmp/ipykernel_1038548/739859072.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_autogluon["timestamp"] = pd.to_datetime(df_autogluon["periodo"].astype(str), format='%Y%m')
/tmp/ipykernel_1038548/739859072.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_autogluon["target"] = df_autogluon["tn+2"]
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250714_231127'
=================== System Info ===================
AutoGluon Version:  

In [19]:
forecast = predictor.predict(ts_data)
forecast_mean = forecast["mean"].reset_index()
resultado = forecast_mean[forecast_mean["timestamp"] == "2020-02-01"]
resultado = resultado[["item_id", "mean"]]
resultado.columns = ["product_id", "tn"]
resultado

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


,product_id,tn
1,20001,1458.205078
3,20002,1056.582520
5,20003,796.555664
7,20004,601.407898
9,20005,637.419495
...,...,...
1551,21263,0.014977
1553,21265,0.061218
1555,21266,0.063014
1557,21267,0.033089


In [20]:
predictions.columns = ["product_id", "tn_regresion"]
resultado = pd.merge(resultado, predictions, on="product_id", how="left")
resultado = resultado.rename(columns={"tn": "tn_autogluon"})
resultado

,product_id,tn_autogluon,tn_regresion
0,20001,1458.205078,1162.707525
1,20002,1056.582520,1183.640604
2,20003,796.555664,684.763931
3,20004,601.407898,580.484961
4,20005,637.419495,563.560780
...,...,...,...
775,21263,0.014977,0.467764
776,21265,0.061218,NaN
777,21266,0.063014,NaN
778,21267,0.033089,NaN


In [22]:
# los valores que son nan en tn_regresson los reemplazo por tn_autogluon
resultado['tn_regresion'] = resultado['tn_regresion'].fillna(resultado['tn_autogluon'])
resultado["tn"] = (resultado["tn_regresion"] + resultado["tn_autogluon"]) / 2

In [23]:
submission = resultado[["product_id", "tn"]]
submission.to_csv("submission_autogluon_regression.csv", index=False)
submission

,product_id,tn
0,20001,1310.456302
1,20002,1120.111562
2,20003,740.659797
3,20004,590.946430
4,20005,600.490137
...,...,...
775,21263,0.241371
776,21265,0.061218
777,21266,0.063014
778,21267,0.033089


In [24]:
submission = resultado[["product_id", "tn_regresion"]]
submission.columns = ["product_id", "tn"]
submission.to_csv("submission_autogluon_regression_tn.csv", index=False)
submission

,product_id,tn
0,20001,1162.707525
1,20002,1183.640604
2,20003,684.763931
3,20004,580.484961
4,20005,563.560780
...,...,...
775,21263,0.467764
776,21265,0.061218
777,21266,0.063014
778,21267,0.033089


In [27]:
df_predicted["predicted_tn+2"].sum()

np.float64(28486.79662313294)

In [26]:
submission = df_predicted[['product_id', 'predicted_tn+2']]
submission.rename(columns={'predicted_tn+2': 'tn'}, inplace=True)
submission.to_csv("submission_ensamble_regressors.csv", index=False)
submission

/tmp/ipykernel_45468/575973912.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission.rename(columns={'predicted_tn+2': 'tn'}, inplace=True)


,product_id,tn
0,20001,1162.707525
1,20002,1183.640604
2,20003,684.763931
3,20004,580.484961
4,20005,563.560780
...,...,...
775,20962,16.941649
776,20975,16.626435
777,20995,16.467131
778,21087,15.038854


In [79]:
import numpy as np

# --- FILTRADO BASE ---
# Solo productos a predecir y mes 201912 (mes_abs == 36)
df_pred = df_agg[
    (df_agg['mes_abs'] == 36) & 
    (df_agg['product_id'].isin(df_780['product_id']))
].copy()

# Determinar qué productos tienen todos los features disponibles
df_pred['completos'] = df_pred[features].notnull().all(axis=1)

# Separar completos e incompletos
df_completos = df_pred[df_pred['completos']].copy()
df_incompletos = df_pred[~df_pred['completos']].copy()

# --- PREDICCIÓN PARA COMPLETOS ---
df_completos['pred'] = model.predict(df_completos[features])
df_completos['pred_tipo'] = 'modelo'

import numpy as np

# --- PREDICCIÓN PARA INCOMPLETOS: promedio ponderado con énfasis en febrero anterior (tn_10) ---

# Pesos manuales: últimos 3 meses, febrero pasado, resto
pesos = np.array([
    3.0,  # tn
    2.5,  # tn_1
    2.0,  # tn_2
    1.5,  # tn_3
    1.5,  # tn_4
    1.5,  # tn_5
    1.0,  # tn_6
    1.0,  # tn_7
    1.0,  # tn_8
    1.0,  # tn_9
    4.0,  # tn_10
    1.0   # tn_11
])

# Dar más peso a tn_10 (correspondiente a mes_abs 26 = 201902)
pesos[10] *= 12  # Aumentar influencia de febrero del año anterior

# Extraer matriz de features
X_incompletos = df_incompletos[features].values

# Crear máscara de valores válidos
máscara_validos = ~np.isnan(X_incompletos)

# Expandir pesos por fila, aplicando la máscara
pesos_expandido = np.tile(pesos, (X_incompletos.shape[0], 1))
pesos_validos = pesos_expandido * máscara_validos

# Calcular promedio ponderado por fila
suma_ponderada = np.nansum(X_incompletos * pesos_validos, axis=1)
suma_pesos = np.nansum(pesos_validos, axis=1)
df_incompletos['pred'] = suma_ponderada / suma_pesos
df_incompletos['pred_tipo'] = 'promedio_ponderado_febrero'


# --- UNIÓN FINAL ---
df_final = pd.concat([df_completos, df_incompletos], axis=0).sort_values('product_id')

# --- VALIDACIÓN DE RESULTADOS ---
print("🔍 Suma de TN a 201912:")
print(f"Completos (modelo): {df_completos['tn'].sum():,.2f}")
print(f"Incompletos (promedio historia): {df_incompletos['tn'].sum():,.2f}")
print(f"Total final: {df_final['tn'].sum():,.2f}")

print("🔍 Suma de predicciones a 202002:")
print(f"Completos (modelo): {df_completos['pred'].sum():,.2f}")
print(f"Incompletos (promedio historia): {df_incompletos['pred'].sum():,.2f}")
print(f"Total final: {df_final['pred'].sum():,.2f}")

print("\n📦 Desglose:")
print(f"Total a predecir: {df_final.shape[0]}  (esperado: {df_780.shape[0]})")
print(f"Completos: {df_completos.shape[0]}  |  Incompletos: {df_incompletos.shape[0]}")
print(df_final['pred_tipo'].value_counts())

# --- VISTA RÁPIDA ---
display(df_final[['product_id', 'periodo', 'pred', 'pred_tipo']].head())



🔍 Suma de TN a 201912:
Completos (modelo): 23,447.86
Incompletos (promedio historia): 1,697.39
Total final: 25,145.25
🔍 Suma de predicciones a 202002:
Completos (modelo): 25,785.02
Incompletos (promedio historia): 1,939.48
Total final: 27,724.50

📦 Desglose:
Total a predecir: 780  (esperado: 780)
Completos: 656  |  Incompletos: 124
pred_tipo
modelo                        656
promedio_ponderado_febrero    124
Name: count, dtype: int64


,product_id,periodo,pred,pred_tipo
35,20001,201912,1162.707525,modelo
71,20002,201912,1183.640604,modelo
107,20003,201912,684.763931,modelo
143,20004,201912,580.484961,modelo
179,20005,201912,563.560780,modelo


In [80]:
import os
from datetime import datetime

# Crear carpeta 'kaggle' si no existe
os.makedirs("kaggle", exist_ok=True)

# Generar timestamp actual
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Nombre de archivo
filename = f"kaggle/predicciones_201912_{timestamp}.csv"

# Exportar CSV
df_final[['product_id', 'pred']].to_csv(filename, index=False)

print(f"Archivo guardado como: {filename}")


Archivo guardado como: kaggle/predicciones_201912_20250703_2151.csv
